# Notebook 02 — Baselines (dictionary + XGBoost)

**What this notebook does:**
1. Load labeled data + splits from Notebook 01
2. Run **dictionary** baseline (re-match libraries at test time)
3. Run **XGBoost** bond-cut baseline (labmate / Ribes-style)
4. Save `outputs/baselines_metrics.json`

All helper code is inlined below (no `src` import).


## 0. Paths

In [ ]:
# Project root = parent of notebooks/ (or cwd if already in Yashi/)
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "Protac_4.0_database_files_downloaded"
PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "outputs"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)


## 1. Helper functions (labelling, reassembly, baselines)

In [ ]:
from __future__ import annotations

import io
import math
import random
from collections import Counter, deque
from dataclasses import dataclass
from typing import Iterable, Optional

import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Draw, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D

RDLogger.DisableLog("rdApp.*")

WARHEAD, LINKER, E3 = 0, 1, 2
CLASS_NAMES = {WARHEAD: "warhead", LINKER: "linker", E3: "e3_ligand"}
CLASS_COLORS = {
    WARHEAD: (1.0, 0.35, 0.35),   # pink
    LINKER:  (0.30, 0.55, 1.0),   # blue
    E3:      (0.30, 0.85, 0.35),  # green
}

# =============================================================================
# 1. Weak labelling  (dictionary + reassembly filter)
# =============================================================================

@dataclass
class RefFragment:
    smiles: str
    mol: Chem.Mol
    n_atoms: int


def prepare_reference_library(
    smiles_list: Iterable[str], min_atoms: int = 5, max_atoms: int = 60
) -> list[RefFragment]:
    """Parse + filter reference SMILES; sort largest-first for greedy match."""
    refs: list[RefFragment] = []
    for smi in smiles_list:
        if not isinstance(smi, str) or not smi:
            continue
        m = Chem.MolFromSmiles(smi)
        if m is None:
            continue
        n = m.GetNumHeavyAtoms()
        if not (min_atoms <= n <= max_atoms):
            continue
        refs.append(RefFragment(Chem.MolToSmiles(m), m, n))
    refs.sort(key=lambda r: r.n_atoms, reverse=True)
    return refs


def _best_match(mol: Chem.Mol, refs: list[RefFragment],
                forbidden: Optional[set[int]] = None) -> Optional[tuple[set[int], str]]:
    """Return (atoms_of_best_match, ref_smiles) or None."""
    forbidden = forbidden or set()
    n_target = mol.GetNumAtoms()
    for ref in refs:
        if ref.n_atoms > n_target:
            continue
        matches = mol.GetSubstructMatches(ref.mol, uniquify=True, useChirality=False)
        for match in matches:
            atoms = set(match)
            if atoms.isdisjoint(forbidden):
                return atoms, ref.smiles
    return None


def _boundary_bonds(mol: Chem.Mol, labels: list[int]) -> list[int]:
    return [b.GetIdx() for b in mol.GetBonds()
            if labels[b.GetBeginAtomIdx()] != labels[b.GetEndAtomIdx()]]


def reassembles(mol: Chem.Mol, labels: list[int]) -> tuple[bool, int]:
    """Cut molecule at boundary bonds; return (success, n_fragments).

    Reassembly succeeds when cutting produces exactly 3 chemically valid
    fragments and no atoms are lost.  This is the Ribes-style validation
    of a splitting, applied to *any* atom-level labelling.
    """
    b_idx = _boundary_bonds(mol, labels)
    if not b_idx:
        return False, 1
    frag_mol = Chem.FragmentOnBonds(mol, b_idx, addDummies=True)
    frags = Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False)
    if len(frags) != 3:
        return False, len(frags)
    atoms_seen = 0
    for f in frags:
        try:
            Chem.SanitizeMol(f)
        except Exception:
            return False, len(frags)
        atoms_seen += sum(1 for a in f.GetAtoms() if a.GetAtomicNum() != 0)
    return (atoms_seen == mol.GetNumAtoms()), len(frags)


def label_protac(smiles: str, wh_refs: list[RefFragment], e3_refs: list[RefFragment],
                 require_reassembly: bool = True) -> Optional[dict]:
    """Weak-label one PROTAC. Return dict or None on failure."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    n = mol.GetNumAtoms()
    if not (15 <= n <= 120):
        return None
    wh = _best_match(mol, wh_refs)
    if wh is None:
        return None
    e3 = _best_match(mol, e3_refs, forbidden=wh[0])
    if e3 is None:
        return None
    labels = [LINKER] * n
    for a in wh[0]:
        labels[a] = WARHEAD
    for a in e3[0]:
        labels[a] = E3
    if LINKER not in labels:
        return None
    if require_reassembly:
        ok, _ = reassembles(mol, labels)
        if not ok:
            return None
    return {
        "smiles": Chem.MolToSmiles(mol),
        "labels": labels,
        "wh_ref": wh[1],
        "e3_ref": e3[1],
        "n_warhead": len(wh[0]),
        "n_linker": labels.count(LINKER),
        "n_e3": len(e3[0]),
    }

# =============================================================================
# 6. Baselines
# =============================================================================

def dictionary_baseline_predict(smiles: str, wh_refs: list[RefFragment],
                                e3_refs: list[RefFragment]) -> Optional[list[int]]:
    """Ribes-style dictionary annotation used as a *baseline*, not as labels.

    We use the exact same routine as weak labelling; a test molecule is
    'predicted' by re-matching against the library at inference time.
    """
    rec = label_protac(smiles, wh_refs, e3_refs, require_reassembly=False)
    return rec["labels"] if rec else None


def evaluate_dictionary_baseline(records: list[dict], indices: list[int],
                                  wh_refs, e3_refs) -> dict:
    hits = exact3 = reasm = 0; total = len(indices); atom_correct = atom_tot = 0
    for idx in indices:
        rec = records[idx]
        pred = dictionary_baseline_predict(rec["smiles"], wh_refs, e3_refs)
        if pred is None:
            atom_tot += len(rec["labels"])
            continue
        hits += 1
        atom_correct += sum(1 for a, b in zip(rec["labels"], pred) if a == b)
        atom_tot += len(rec["labels"])
        mol = Chem.MolFromSmiles(rec["smiles"])
        ok, n = reassembles(mol, pred)
        exact3 += (n == 3); reasm += ok
    return {
        "coverage":         hits / max(total, 1),
        "atom_acc":         atom_correct / max(atom_tot, 1),
        "exact_3_frag":     exact3 / max(total, 1),
        "reassembly":       reasm / max(total, 1),
        "n_test_molecules": total,
    }


# ---- Ayush-style XGBoost bond-cutting baseline -----------------------------

def bond_feature_row(mol: Chem.Mol, bond: Chem.Bond, bc_scores: dict[int, float],
                     morgan_svd: np.ndarray) -> np.ndarray:
    """Compact 18-feature bond descriptor + Morgan SVD context."""
    i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    a1, a2 = mol.GetAtomWithIdx(i), mol.GetAtomWithIdx(j)
    is_ring = int(bond.IsInRing())
    sp3 = 0.5 * (int(a1.GetHybridization() == Chem.HybridizationType.SP3)
                 + int(a2.GetHybridization() == Chem.HybridizationType.SP3))
    rotatable = int((not is_ring) and bond.GetBondType() == Chem.BondType.SINGLE
                    and a1.GetDegree() > 1 and a2.GetDegree() > 1)
    bc = bc_scores.get(bond.GetIdx(), 0.0)
    is_amide = int((a1.GetAtomicNum() == 7 and a2.GetAtomicNum() == 6
                    and any(nb.GetAtomicNum() == 8 and mol.GetBondBetweenAtoms(a2.GetIdx(), nb.GetIdx()).GetBondType() == Chem.BondType.DOUBLE for nb in a2.GetNeighbors()))
                   or (a2.GetAtomicNum() == 7 and a1.GetAtomicNum() == 6
                    and any(nb.GetAtomicNum() == 8 and mol.GetBondBetweenAtoms(a1.GetIdx(), nb.GetIdx()).GetBondType() == Chem.BondType.DOUBLE for nb in a1.GetNeighbors())))
    is_ether = int(a1.GetAtomicNum() == 8 or a2.GetAtomicNum() == 8)
    local = np.array([
        is_ring, sp3, rotatable, bc,
        a1.GetAtomicNum(), a2.GetAtomicNum(),
        {Chem.BondType.SINGLE: 1.0, Chem.BondType.DOUBLE: 2.0,
         Chem.BondType.TRIPLE: 3.0, Chem.BondType.AROMATIC: 1.5}.get(bond.GetBondType(), 1.0),
        is_amide, is_ether,
        int(a1.IsInRing()), int(a2.IsInRing()),
        int(a1.GetIsAromatic()), int(a2.GetIsAromatic()),
        int(bond.GetIsAromatic()),
        a1.GetDegree(), a2.GetDegree(),
        int(a1.GetFormalCharge()), int(a2.GetFormalCharge()),
    ], dtype=np.float32)
    return np.concatenate([local, morgan_svd])


def approx_betweenness(mol: Chem.Mol) -> dict[int, float]:
    """Cheap BC approximation on the bond graph (Ayush's bridge-bond trick)."""
    G = nx.Graph()
    for a in mol.GetAtoms():
        G.add_node(a.GetIdx())
    for b in mol.GetBonds():
        G.add_edge(b.GetBeginAtomIdx(), b.GetEndAtomIdx(), idx=b.GetIdx())
    n = G.number_of_nodes()
    out: dict[int, float] = {}
    if n < 2:
        return out
    denom = n * (n - 1) / 2
    bridges = set(nx.bridges(G)) if n > 1 else set()
    for b in mol.GetBonds():
        u, v = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        if (u, v) in bridges or (v, u) in bridges:
            H = G.copy(); H.remove_edge(u, v)
            comps = list(nx.connected_components(H))
            if len(comps) == 2:
                a, b_sz = len(comps[0]), len(comps[1])
                out[b.GetIdx()] = (a * b_sz) / denom
    return out


def build_xgb_matrix(records: list[dict], indices: list[int],
                     morgan_svd_all: np.ndarray) -> tuple[np.ndarray, np.ndarray, list[int]]:
    """Return X (bonds x F), y (0/1 cut bond), groups (protac ids)."""
    X_rows, y_rows, grp_rows = [], [], []
    for pos, idx in enumerate(indices):
        rec = records[idx]
        mol = Chem.MolFromSmiles(rec["smiles"])
        bc = approx_betweenness(mol)
        cut_set = set(_boundary_bonds(mol, rec["labels"]))
        svd_vec = morgan_svd_all[idx]
        for b in mol.GetBonds():
            X_rows.append(bond_feature_row(mol, b, bc, svd_vec))
            y_rows.append(1 if b.GetIdx() in cut_set else 0)
            grp_rows.append(idx)
    return np.stack(X_rows), np.array(y_rows), grp_rows


def compute_morgan_svd(records: list[dict], n_components: int = 32) -> np.ndarray:
    """Global molecule-level SVD-compressed Morgan fingerprints (per PROTAC)."""
    fps = np.zeros((len(records), 256), dtype=np.float32)
    for i, r in enumerate(records):
        mol = Chem.MolFromSmiles(r["smiles"])
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=256)
        arr = np.zeros(256, dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        fps[i] = arr.astype(np.float32)
    from sklearn.decomposition import TruncatedSVD
    n_comp = min(n_components, fps.shape[1] - 1, len(records) - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=0)
    return svd.fit_transform(fps)


def xgb_baseline_predict_and_score(records: list[dict], train_idx: list[int],
                                   test_idx: list[int], morgan_svd_all: np.ndarray):
    """Train XGBoost on train bonds, predict top-2 cuts per test PROTAC,
    then convert cuts into a W/L/E3 atom labelling to measure the same metrics."""
    from xgboost import XGBClassifier
    X_tr, y_tr, _ = build_xgb_matrix(records, train_idx, morgan_svd_all)
    X_te, y_te, grp_te = build_xgb_matrix(records, test_idx, morgan_svd_all)
    pos = max(y_tr.sum(), 1); neg = max((y_tr == 0).sum(), 1)
    clf = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                        tree_method="hist", scale_pos_weight=float(neg) / pos,
                        n_jobs=-1, verbosity=0, eval_metric="logloss")
    clf.fit(X_tr, y_tr)
    scores = clf.predict_proba(X_te)[:, 1]
    # top-2 cuts per PROTAC
    exact3 = reasm = atom_correct = atom_tot = 0
    for idx in test_idx:
        rec = records[idx]
        mol = Chem.MolFromSmiles(rec["smiles"])
        # gather this molecule's bond scores
        mol_scores = []
        for pos_i, gid in enumerate(grp_te):
            if gid == idx:
                mol_scores.append((pos_i, scores[pos_i]))
        # pick top-2 bonds
        top2 = sorted(mol_scores, key=lambda t: -t[1])[:2]
        # map bond global row -> bond index in this molecule
        local_bonds = list(range(mol.GetNumBonds()))
        # since build_xgb_matrix visits bonds in order, mapping is trivial
        first_row_for_mol = next(i for i, g in enumerate(grp_te) if g == idx)
        cut_bond_indices = [t[0] - first_row_for_mol for t in top2]
        pred_labels = xgb_cuts_to_labels(mol, cut_bond_indices)
        atom_correct += sum(1 for a, b in zip(rec["labels"], pred_labels) if a == b)
        atom_tot += len(pred_labels)
        ok, n = reassembles(mol, pred_labels)
        exact3 += (n == 3); reasm += ok
    return {
        "atom_acc":         atom_correct / max(atom_tot, 1),
        "exact_3_frag":     exact3 / max(len(test_idx), 1),
        "reassembly":       reasm / max(len(test_idx), 1),
        "n_test_molecules": len(test_idx),
        "bond_roc_auc":     _pr_auc(y_te.tolist(), scores.tolist()),
    }, clf


def xgb_cuts_to_labels(mol: Chem.Mol, cut_bond_indices: list[int]) -> list[int]:
    """Convert 2 cut bonds into atom W/L/E3 labels.

    The three resulting fragments are: the two 'outside' fragments become
    warhead / E3 and the fragment in the middle (connected to both cuts)
    becomes linker.  If cutting fails to give 3 pieces, label everything
    as linker so the metric penalises us.
    """
    n = mol.GetNumAtoms()
    if not cut_bond_indices:
        return [LINKER] * n
    rw = Chem.RWMol(mol)
    for b in cut_bond_indices:
        try:
            bond = mol.GetBondWithIdx(b)
            rw.RemoveBond(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())
        except Exception:
            pass
    frag_atoms = Chem.GetMolFrags(rw.GetMol(), asMols=False)
    if len(frag_atoms) != 3:
        return [LINKER] * n
    # linker = fragment containing atoms from BOTH cut bonds (touches both cuts)
    touches: list[set[int]] = []
    cut_atom_pairs = []
    for b in cut_bond_indices:
        bond = mol.GetBondWithIdx(b)
        cut_atom_pairs.append((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
    for frag in frag_atoms:
        s = set(frag)
        touches.append(s)
    # find fragment that shares atoms with BOTH cut bonds
    linker_idx = -1
    for k, s in enumerate(touches):
        if all(any(a in s for a in pair) for pair in cut_atom_pairs):
            linker_idx = k; break
    if linker_idx == -1:
        # fallback: smallest fragment = linker
        linker_idx = min(range(3), key=lambda k: len(touches[k]))
    remaining = [k for k in range(3) if k != linker_idx]
    # assign W to larger of the two remaining fragments (arbitrary but stable)
    remaining.sort(key=lambda k: -len(touches[k]))
    labels = [LINKER] * n
    for a in touches[remaining[0]]:
        labels[a] = WARHEAD
    for a in touches[remaining[1]]:
        labels[a] = E3
    return labels


# small metric helper used by XGBoost scoring (kept here so this notebook stays self-contained)
def _pr_auc(y_true, scores):
    if not y_true:
        return 0.0
    order = sorted(range(len(scores)), key=lambda i: -scores[i])
    P = sum(y_true)
    if P == 0:
        return 0.0
    tp = fp = 0
    ap = 0.0
    prev_recall = 0.0
    for i in order:
        if y_true[i]:
            tp += 1
        else:
            fp += 1
        prec = tp / (tp + fp)
        rec = tp / P
        ap += prec * (rec - prev_recall)
        prev_recall = rec
    return ap


## 2. Load saved data

In [ ]:
import json, pickle

records = [json.loads(l) for l in open(PROCESSED / "labeled_protacs.jsonl")]
splits  = pickle.load(open(PROCESSED / "splits.pkl", "rb"))
refs    = pickle.load(open(PROCESSED / "refs.pkl", "rb"))
wh_refs = prepare_reference_library(refs["warheads"])
e3_refs = prepare_reference_library(refs["e3"])
print("records:", len(records), " splits:", list(splits))


## 3. Dictionary baseline on every split

In [ ]:
print("=== dictionary baseline ===")
dict_metrics = {}
for sname, s in splits.items():
    m = evaluate_dictionary_baseline(records, s["test"], wh_refs, e3_refs)
    dict_metrics[sname] = m
    print(f"  {sname:18s}  atom={m['atom_acc']:.3f}  exact3={m['exact_3_frag']:.3f}  reasm={m['reassembly']:.3f}")


## 4. XGBoost bond-cut baseline on every split

In [ ]:
print("=== XGBoost baseline ===")
morgan_svd = compute_morgan_svd(records, n_components=32)
xgb_metrics = {}
for sname, s in splits.items():
    print("---", sname, "---")
    m, clf = xgb_baseline_predict_and_score(records, s["train"], s["test"], morgan_svd)
    xgb_metrics[sname] = m
    print(f"  atom={m['atom_acc']:.3f}  exact3={m['exact_3_frag']:.3f}  reasm={m['reassembly']:.3f}")


## 5. Save metrics

In [ ]:
import json
with open(OUT / "baselines_metrics.json", "w") as f:
    json.dump({"dictionary": dict_metrics, "xgboost": xgb_metrics}, f, indent=2)
print("saved", OUT / "baselines_metrics.json")
print("DONE — Notebook 02 complete.")
